In [1]:
import pandas as pd
from pathlib import Path

# 프로젝트 경로
ROOT_DIR = Path.cwd().parent
PROCESSED_DIR = ROOT_DIR / "data" / "processed"

# CSV 파일 불러오기
elderly_df = pd.read_csv(
    PROCESSED_DIR / "65세이상_고령인구_정제.csv",
    encoding="utf-8-sig"
)

single_df = pd.read_csv(
    PROCESSED_DIR / "노인_1인가구_정제.csv",
    encoding="utf-8-sig"
)

climate_df = pd.read_csv(
    PROCESSED_DIR / "시군구_기후데이터_정제.csv",
    encoding="utf-8-sig"
)

shelter_df = pd.read_csv(
    PROCESSED_DIR / "쉼터_지역별_통합_정제.csv",
    encoding="utf-8-sig"
)

print("고령인구:", elderly_df.shape)
print("노인 1인가구:", single_df.shape)
print("기후 데이터:", climate_df.shape)
print("쉼터 데이터:", shelter_df.shape)

고령인구: (229, 7)
노인 1인가구: (229, 7)
기후 데이터: (1145, 19)
쉼터 데이터: (229, 4)


In [2]:
# 각 데이터셋의 컬럼명 및 데이터 타입 확인

dataframes = {
    "고령인구": elderly_df,
    "노인 1인가구": single_df,
    "기후 데이터": climate_df,
    "쉼터 데이터": shelter_df
}

for name, df in dataframes.items():
    print(f"\n{'=' * 50}")
    print(f"[{name}]")
    print("컬럼명:", df.columns.tolist())
    print("\n데이터 타입:")
    print(df.dtypes)


[고령인구]
컬럼명: ['시도', '행정구역(시군구)별', '2021 년', '2022 년', '2023 년', '2024 년', '2025 년']

데이터 타입:
시도                str
행정구역(시군구)별        str
2021 년        float64
2022 년        float64
2023 년        float64
2024 년        float64
2025 년        float64
dtype: object

[노인 1인가구]
컬럼명: ['시도', '행정구역별(시군구)', '2021', '2022', '2023', '2024', '2025']

데이터 타입:
시도                str
행정구역별(시군구)        str
2021          float64
2022          float64
2023          float64
2024          float64
2025          float64
dtype: object

[기후 데이터]
컬럼명: ['시도', '시군구', '경도', '위도', '관측소_지점', '관측소명', '관측소_시도', '거리_km', '연도', '폭염일수', '관측일수_x', '여름관측일수', '여름부족일수', '여름관측_주의', '한파일수', '관측일수_y', '겨울관측일수', '겨울부족일수', '겨울관측_주의']

데이터 타입:
시도             str
시군구            str
경도         float64
위도         float64
관측소_지점       int64
관측소명           str
관측소_시도         str
거리_km      float64
연도           int64
폭염일수         int64
관측일수_x       int64
여름관측일수       int64
여름부족일수       int64
여름관측_주의       bool
한파일수         int64
관측일수_y   

In [3]:
# 1. 시군구 컬럼명 통일
elderly_df = elderly_df.rename(
    columns={"행정구역(시군구)별": "시군구"}
)

single_df = single_df.rename(
    columns={"행정구역별(시군구)": "시군구"}
)

# 2. 연도 컬럼명 통일
elderly_df = elderly_df.rename(columns={
    "2021 년": "2021",
    "2022 년": "2022",
    "2023 년": "2023",
    "2024 년": "2024",
    "2025 년": "2025"
})

# 3. 행정구역 명칭 통일
province_mapping = {
    "광주광역시": "전남광주통합특별시",
    "전라남도": "전남광주통합특별시"
}

district_mapping = {
    "창원시(통합)": "창원시",
    "세종특별자치시": "세종시"
}

for df in [elderly_df, single_df, climate_df, shelter_df]:
    df["시도"] = df["시도"].replace(province_mapping)
    df["시군구"] = df["시군구"].replace(district_mapping)

# 4. 결과 확인
print("[고령인구]")
print(elderly_df.columns.tolist())

print("\n[노인 1인가구]")
print(single_df.columns.tolist())

print("\n[행정구역 개수]")
for name, df in {
    "고령인구": elderly_df,
    "노인 1인가구": single_df,
    "기후 데이터": climate_df,
    "쉼터 데이터": shelter_df
}.items():
    print(f"{name}: {df[['시도', '시군구']].drop_duplicates().shape[0]}개")

[고령인구]
['시도', '시군구', '2021', '2022', '2023', '2024', '2025']

[노인 1인가구]
['시도', '시군구', '2021', '2022', '2023', '2024', '2025']

[행정구역 개수]
고령인구: 229개
노인 1인가구: 229개
기후 데이터: 229개
쉼터 데이터: 229개


In [4]:
# 연도별 고령인구 데이터 세로형 변환
elderly_long = elderly_df.melt(
    id_vars=["시도", "시군구"],
    var_name="연도",
    value_name="고령인구"
)

# 연도별 노인 1인가구 데이터 세로형 변환
single_long = single_df.melt(
    id_vars=["시도", "시군구"],
    var_name="연도",
    value_name="노인_1인가구"
)

# 연도 데이터 타입 통일
elderly_long["연도"] = elderly_long["연도"].astype(int)
single_long["연도"] = single_long["연도"].astype(int)

# 결과 확인
print("[고령인구]")
print(elderly_long.shape)
display(elderly_long.head())

print("\n[노인 1인가구]")
print(single_long.shape)
display(single_long.head())

[고령인구]
(1145, 4)


,시도,시군구,연도,고령인구
0,강원특별자치도,강릉시,2021,47318.0
1,강원특별자치도,고성군,2021,8236.0
2,강원특별자치도,동해시,2021,19039.0
3,강원특별자치도,삼척시,2021,16884.0
4,강원특별자치도,속초시,2021,16673.0



[노인 1인가구]
(1145, 4)


,시도,시군구,연도,노인_1인가구
0,강원특별자치도,강릉시,2021,10813.0
1,강원특별자치도,고성군,2021,2006.0
2,강원특별자치도,동해시,2021,4615.0
3,강원특별자치도,삼척시,2021,4335.0
4,강원특별자치도,속초시,2021,4313.0


In [5]:
# 고령인구 + 노인 1인가구 병합
welfare_df = pd.merge(
    elderly_long,
    single_long,
    on=["시도", "시군구", "연도"],
    how="inner",
    validate="one_to_one"
)

# 결과 확인
print("통합 데이터 크기:", welfare_df.shape)
print("\n컬럼:", welfare_df.columns.tolist())

print("\n샘플 데이터")
display(welfare_df.head(10))

# 중복 및 결측치 확인
print("\n중복 행 개수:", welfare_df.duplicated(
    subset=["시도", "시군구", "연도"]
).sum())

print("\n결측치")
print(welfare_df.isnull().sum())

통합 데이터 크기: (1145, 5)

컬럼: ['시도', '시군구', '연도', '고령인구', '노인_1인가구']

샘플 데이터


,시도,시군구,연도,고령인구,노인_1인가구
0,강원특별자치도,강릉시,2021,47318.0,10813.0
1,강원특별자치도,고성군,2021,8236.0,2006.0
2,강원특별자치도,동해시,2021,19039.0,4615.0
3,강원특별자치도,삼척시,2021,16884.0,4335.0
4,강원특별자치도,속초시,2021,16673.0,4313.0
5,강원특별자치도,양구군,2021,4893.0,1026.0
6,강원특별자치도,양양군,2021,8671.0,2077.0
7,강원특별자치도,영월군,2021,11735.0,3082.0
8,강원특별자치도,원주시,2021,56771.0,12454.0
9,강원특별자치도,인제군,2021,6899.0,1571.0



중복 행 개수: 0

결측치
시도         0
시군구        0
연도         0
고령인구       0
노인_1인가구    0
dtype: int64


In [6]:
# 복지 데이터 + 기후 데이터 병합
integrated_df = pd.merge(
    welfare_df,
    climate_df,
    on=["시도", "시군구", "연도"],
    how="inner",
    validate="one_to_one"
)

# 결과 확인
print("통합 데이터 크기:", integrated_df.shape)

print("\n컬럼 목록")
print(integrated_df.columns.tolist())

print("\n샘플 데이터")
display(integrated_df.head())

print("\n중복 행 개수:", integrated_df.duplicated(
    subset=["시도", "시군구", "연도"]
).sum())

print("\n결측치")
print(integrated_df.isnull().sum())

통합 데이터 크기: (1145, 21)

컬럼 목록
['시도', '시군구', '연도', '고령인구', '노인_1인가구', '경도', '위도', '관측소_지점', '관측소명', '관측소_시도', '거리_km', '폭염일수', '관측일수_x', '여름관측일수', '여름부족일수', '여름관측_주의', '한파일수', '관측일수_y', '겨울관측일수', '겨울부족일수', '겨울관측_주의']

샘플 데이터


,시도,시군구,연도,고령인구,노인_1인가구,경도,위도,관측소_지점,관측소명,관측소_시도,...,폭염일수,관측일수_x,여름관측일수,여름부족일수,여름관측_주의,한파일수,관측일수_y,겨울관측일수,겨울부족일수,겨울관측_주의
0,강원특별자치도,강릉시,2021,47318.0,10813.0,128.883209,37.709604,105,강릉,강원특별자치도,...,11,365,122,0,False,4,365,151,0,False
1,강원특별자치도,고성군,2021,8236.0,2006.0,128.358361,38.401009,90,속초,강원특별자치도,...,4,365,122,0,False,7,365,151,0,False
2,강원특별자치도,동해시,2021,19039.0,4615.0,129.045397,37.515759,106,동해,강원특별자치도,...,0,365,122,0,False,2,365,151,0,False
3,강원특별자치도,삼척시,2021,16884.0,4335.0,129.173724,37.257882,216,태백,강원특별자치도,...,0,365,122,0,False,19,365,151,0,False
4,강원특별자치도,속초시,2021,16673.0,4313.0,128.515038,38.175014,90,속초,강원특별자치도,...,4,365,122,0,False,7,365,151,0,False



중복 행 개수: 0

결측치
시도         0
시군구        0
연도         0
고령인구       0
노인_1인가구    0
경도         0
위도         0
관측소_지점     0
관측소명       0
관측소_시도     0
거리_km      0
폭염일수       0
관측일수_x     0
여름관측일수     0
여름부족일수     0
여름관측_주의    0
한파일수       0
관측일수_y     0
겨울관측일수     0
겨울부족일수     0
겨울관측_주의    0
dtype: int64


In [7]:
# 기후·복지 통합 데이터 + 쉼터 데이터 병합
final_df = pd.merge(
    integrated_df,
    shelter_df,
    on=["시도", "시군구"],
    how="inner",
    validate="many_to_one"
)

# 결과 확인
print("최종 데이터 크기:", final_df.shape)

print("\n컬럼 목록")
print(final_df.columns.tolist())

print("\n샘플 데이터")
display(final_df.head())

print("\n중복 지역·연도:")
print(final_df.duplicated(
    subset=["시도", "시군구", "연도"]
).sum())

print("\n결측치:")
print(final_df.isnull().sum())

최종 데이터 크기: (1145, 23)

컬럼 목록
['시도', '시군구', '연도', '고령인구', '노인_1인가구', '경도', '위도', '관측소_지점', '관측소명', '관측소_시도', '거리_km', '폭염일수', '관측일수_x', '여름관측일수', '여름부족일수', '여름관측_주의', '한파일수', '관측일수_y', '겨울관측일수', '겨울부족일수', '겨울관측_주의', '무더위쉼터_개수', '한파쉼터_개수']

샘플 데이터


,시도,시군구,연도,고령인구,노인_1인가구,경도,위도,관측소_지점,관측소명,관측소_시도,...,여름관측일수,여름부족일수,여름관측_주의,한파일수,관측일수_y,겨울관측일수,겨울부족일수,겨울관측_주의,무더위쉼터_개수,한파쉼터_개수
0,강원특별자치도,강릉시,2021,47318.0,10813.0,128.883209,37.709604,105,강릉,강원특별자치도,...,122,0,False,4,365,151,0,False,193,202
1,강원특별자치도,고성군,2021,8236.0,2006.0,128.358361,38.401009,90,속초,강원특별자치도,...,122,0,False,7,365,151,0,False,28,28
2,강원특별자치도,동해시,2021,19039.0,4615.0,129.045397,37.515759,106,동해,강원특별자치도,...,122,0,False,2,365,151,0,False,157,157
3,강원특별자치도,삼척시,2021,16884.0,4335.0,129.173724,37.257882,216,태백,강원특별자치도,...,122,0,False,19,365,151,0,False,63,63
4,강원특별자치도,속초시,2021,16673.0,4313.0,128.515038,38.175014,90,속초,강원특별자치도,...,122,0,False,7,365,151,0,False,124,58



중복 지역·연도:
0

결측치:
시도          0
시군구         0
연도          0
고령인구        0
노인_1인가구     0
경도          0
위도          0
관측소_지점      0
관측소명        0
관측소_시도      0
거리_km       0
폭염일수        0
관측일수_x      0
여름관측일수      0
여름부족일수      0
여름관측_주의     0
한파일수        0
관측일수_y      0
겨울관측일수      0
겨울부족일수      0
겨울관측_주의     0
무더위쉼터_개수    0
한파쉼터_개수     0
dtype: int64


In [8]:
# 최종 데이터 저장 경로
FINAL_DIR = ROOT_DIR / "data" / "final"
FINAL_DIR.mkdir(parents=True, exist_ok=True)

FINAL_PATH = FINAL_DIR / "시군구_기후복지_통합데이터.csv"

# CSV 저장
final_df.to_csv(
    FINAL_PATH,
    index=False,
    encoding="utf-8-sig"
)

print("저장 완료!")
print("파일 경로:", FINAL_PATH)
print("데이터 크기:", final_df.shape)

저장 완료!
파일 경로: c:\Users\expri\Desktop\SafeWeather-AI\data\final\시군구_기후복지_통합데이터.csv
데이터 크기: (1145, 23)
